# Phase 3.2 — Content-Based Recommendation

Build a content-based product recommender from the available Retailrocket product metadata.

The raw metadata uses generic `property` / `value` fields, so product representations are created from those available metadata tokens rather than inventing unavailable attributes.


In [11]:
from pathlib import Path
from collections import defaultdict

import numpy as np
import pandas as pd

from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.neighbors import NearestNeighbors

PROJECT_ROOT = Path.cwd()
while PROJECT_ROOT != PROJECT_ROOT.parent:
    if (PROJECT_ROOT / "data" / "processed").exists():
        break
    PROJECT_ROOT = PROJECT_ROOT.parent

RAW_DIR = PROJECT_ROOT / "data" / "raw"
PROCESSED_DIR = PROJECT_ROOT / "data" / "processed"

PROPERTIES_1 = RAW_DIR / "item_properties_part1.csv"
PROPERTIES_2 = RAW_DIR / "item_properties_part2.csv"

TRAIN_PATH = PROCESSED_DIR / "train_interactions.csv"
TEST_PATH = PROCESSED_DIR / "test_interactions.csv"

print("Project root:", PROJECT_ROOT)


Project root: f:\annuspeaks.com\recommendation-system


## 3.2.1 Build Product Representations

Combine each product's available property/value pairs into a text representation.

Example:

`property_1=value_a property_2=value_b ...`

This representation is suitable for TF-IDF similarity.


In [12]:
# Read and clean product metadata incrementally.

parts = []

for path in [PROPERTIES_1, PROPERTIES_2]:
    for chunk in pd.read_csv(
        path,
        usecols=["itemid", "property", "value"],
        chunksize=250_000,
    ):
        chunk = chunk.dropna(subset=["itemid", "property", "value"]).copy()
        chunk["property"] = chunk["property"].astype(str).str.strip()
        chunk["value"] = chunk["value"].astype(str).str.strip()
        chunk = chunk[(chunk["property"] != "") & (chunk["value"] != "")]
        parts.append(chunk)

properties = pd.concat(parts, ignore_index=True).drop_duplicates(
    subset=["itemid", "property", "value"]
)

print("Clean property records:", f"{len(properties):,}")


Clean property records: 12,778,737


In [13]:
# Create one text representation per product.

properties["token"] = (
    properties["property"]
    + "="
    + properties["value"]
)

product_text = (
    properties.groupby("itemid")["token"]
    .agg(" ".join)
    .reset_index()
    .rename(columns={"itemid": "item_id"})
)

print("Products with metadata representation:", f"{len(product_text):,}")
display(product_text.head())


Products with metadata representation: 417,053


,item_id,token
0,0,112=679677 283=66094 372274 478989 227=1152934...
1,1,296=866110 59=769062 813=814966 33=1128577 100...
2,2,282=n192.000 145688 332=n72.000 159=519769 283...
3,3,159=519769 available=0 678=327918 1080=769062 ...
4,4,available=0 115=n24.000 897=324209 28=150169 1...


## 3.2.2 TF-IDF Product Representation

Convert product metadata into a sparse TF-IDF matrix.

A bounded vocabulary keeps the representation practical for the large catalog.


In [14]:
vectorizer = TfidfVectorizer(
    lowercase=True,
    token_pattern=r"(?u)\b\w+=\S+\b",
    min_df=2,
    max_features=50_000,
)

product_matrix = vectorizer.fit_transform(product_text["token"])

print("Matrix shape:", product_matrix.shape)
print("Non-zero values:", f"{product_matrix.nnz:,}")


Matrix shape: (417053, 50000)
Non-zero values: 10,810,918


## 3.2.3 Similarity Search

Use cosine similarity through a nearest-neighbor index over the sparse product representation.


In [15]:
# Build the similarity index.

similarity_index = NearestNeighbors(
    metric="cosine",
    algorithm="brute",
    n_neighbors=21,
    n_jobs=-1,
)

similarity_index.fit(product_matrix)

item_to_index = {
    item_id: idx
    for idx, item_id in enumerate(product_text["item_id"])
}

index_to_item = product_text["item_id"].to_numpy()

print("Similarity index ready.")


Similarity index ready.


In [16]:
def similar_products(item_id, k=10):
    if item_id not in item_to_index:
        return []

    idx = item_to_index[item_id]

    distances, indices = similarity_index.kneighbors(
        product_matrix[idx],
        n_neighbors=min(k + 1, len(index_to_item)),
    )

    recommendations = []

    for distance, neighbor_idx in zip(distances[0], indices[0]):
        candidate = index_to_item[neighbor_idx]

        if candidate == item_id:
            continue

        recommendations.append({
            "item_id": candidate,
            "similarity": 1.0 - float(distance),
        })

        if len(recommendations) == k:
            break

    return recommendations

example_item = int(index_to_item[0])

print("Example item:", example_item)
display(pd.DataFrame(similar_products(example_item, 10)))


Example item: 0


,item_id,similarity
0,350400,0.859227
1,4200,0.858755
2,38532,0.817537
3,56724,0.817277
4,102722,0.816304
5,142772,0.814602
6,9494,0.814317
7,274674,0.777016
8,94796,0.745628
9,447959,0.740394


## 3.2.4 Personalized Content-Based Recommendations

Build a user profile from the products the user interacted with in training.

The profile is the weighted average of the user's interacted product vectors. Recommendations are products with the highest cosine similarity to that profile.


In [17]:
# Load training interactions.

train = pd.read_csv(
    TRAIN_PATH,
    usecols=["user_id", "item_id", "weight", "timestamp"],
)

test = pd.read_csv(
    TEST_PATH,
    usecols=["user_id", "item_id"],
)

print("Train interactions:", f"{len(train):,}")
print("Test users:", f"{test['user_id'].nunique():,}")


Train interactions: 2,356,045
Test users: 200,028


In [18]:
def content_recommend(user_id, k=10):
    user_rows = train[train["user_id"] == user_id]

    if user_rows.empty:
        return []

    profile_items = user_rows.sort_values("timestamp").tail(20)

    valid = profile_items[
        profile_items["item_id"].isin(item_to_index)
    ]

    if valid.empty:
        return []

    indices = [
        item_to_index[item_id]
        for item_id in valid["item_id"]
    ]

    weights = valid["weight"].to_numpy(dtype=float)

    # Weighted user profile.
    weighted = product_matrix[indices].multiply(weights[:, None])
    profile = weighted.sum(axis=0)
    profile = np.asarray(profile).ravel()

    norm = np.linalg.norm(profile)

    if norm == 0:
        return []

    profile = profile / norm

    # Sparse matrix × dense profile returns a dense ndarray in this setup.
    similarities = np.asarray(
        product_matrix.dot(profile)
    ).ravel()

    seen = set(valid["item_id"])
    ranked_indices = np.argsort(-similarities)

    recommendations = []

    for idx in ranked_indices:
        candidate = index_to_item[idx]

        if candidate in seen:
            continue

        recommendations.append({
            "item_id": candidate,
            "similarity": float(similarities[idx]),
        })

        if len(recommendations) == k:
            break

    return recommendations

example_user = int(train["user_id"].iloc[0])

print("Example user:", example_user)
display(pd.DataFrame(content_recommend(example_user, 10)))


Example user: 0


,item_id,similarity
0,45786,0.758383
1,404151,0.710097
2,169503,0.697910
3,8692,0.628196
4,26171,0.495918
5,77620,0.495918
6,35327,0.495414
7,127865,0.494163
8,180446,0.493144
9,373467,0.491498


## 3.2.5 Evaluate Content-Based Recommendations

Evaluate HitRate@10 on a small deterministic sample using vectorized batch scoring.

Instead of calculating a full catalog score separately for every user, user profiles are built in batches and the catalog is scored in chunks.


In [19]:
# Deterministic evaluation sample

test_targets = (
    test.groupby("user_id")["item_id"]
    .last()
    .to_dict()
)

MAX_EVAL_USERS = 200
eval_users = sorted(test_targets)[:MAX_EVAL_USERS]

print("Evaluation users:", len(eval_users))


Evaluation users: 200


In [20]:
# Fast vectorized HitRate@10 evaluation

catalog_matrix = product_matrix
catalog_items = index_to_item

hits = 0
evaluated = 0

for start in range(0, len(eval_users), 25):
    batch_users = eval_users[start:start + 25]
    profiles = []
    valid_users = []

    for user_id in batch_users:
        user_rows = train[train["user_id"] == user_id]

        valid = (
            user_rows
            .sort_values("timestamp")
            .tail(20)
        )
        valid = valid[valid["item_id"].isin(item_to_index)]

        if valid.empty:
            continue

        indices = [item_to_index[x] for x in valid["item_id"]]
        weights = valid["weight"].to_numpy(dtype=float)

        profile = np.asarray(
            product_matrix[indices].multiply(weights[:, None]).sum(axis=0)
        ).ravel()

        norm = np.linalg.norm(profile)

        if norm == 0:
            continue

        profiles.append(profile / norm)
        valid_users.append((user_id, set(valid["item_id"])))

    if not profiles:
        continue

    profile_matrix = np.vstack(profiles)

    # Score catalog in chunks to keep memory bounded.
    scores = np.zeros(
        (len(profiles), len(catalog_items)),
        dtype=np.float32
    )

    for c_start in range(0, catalog_matrix.shape[0], 50_000):
        c_end = min(c_start + 50_000, catalog_matrix.shape[0])
        scores[:, c_start:c_end] = (
            profile_matrix @ catalog_matrix[c_start:c_end].T
        ).astype(np.float32)

    for row_idx, (user_id, seen) in enumerate(valid_users):
        row_scores = scores[row_idx].copy()

        # Exclude products already interacted with.
        for item_id in seen:
            idx = item_to_index.get(item_id)
            if idx is not None:
                row_scores[idx] = -np.inf

        top_indices = np.argpartition(
            row_scores,
            -10
        )[-10:]

        target = test_targets[user_id]

        if target in set(catalog_items[top_indices]):
            hits += 1

        evaluated += 1

print("Evaluated users:", evaluated)
print("Hits:", hits)

content_hit_rate = (
    hits / evaluated
    if evaluated
    else 0.0
)

content_result = pd.DataFrame([{
    "model": "Content-Based",
    "K": 10,
    "evaluated_users": evaluated,
    "hits": hits,
    "HitRate@10": content_hit_rate,
}])

display(content_result)


Evaluated users: 197
Hits: 7


,model,K,evaluated_users,hits,HitRate@10
0,Content-Based,10,197,7,0.035533


## 3.2.6 New-Product Support

A product does not need historical interactions to receive content-based similar-product recommendations.

If a product has metadata, its TF-IDF representation can be indexed and compared with the catalog.


In [21]:
# Identify products with metadata but no observed training interaction.

train_items = set(train["item_id"].unique())
metadata_items = set(product_text["item_id"].unique())

new_metadata_items = sorted(metadata_items - train_items)

print(
    "Metadata products without training interactions:",
    f"{len(new_metadata_items):,}"
)

if new_metadata_items:
    new_product = new_metadata_items[0]
    print("Example new product:", new_product)
    display(pd.DataFrame(similar_products(new_product, 10)))
else:
    print("No metadata-only products found.")


Metadata products without training interactions: 237,159
Example new product: 0


,item_id,similarity
0,350400,0.859227
1,4200,0.858755
2,38532,0.817537
3,56724,0.817277
4,102722,0.816304
5,142772,0.814602
6,9494,0.814317
7,274674,0.777016
8,94796,0.745628
9,447959,0.740394


## Phase 3.2 Completion

- Product representations built from available metadata.
- TF-IDF similarity search built.
- Personalized content-based Top-K recommendations implemented.
- Content-based HitRate@10 evaluated on a deterministic test-user sample.
- Metadata-only/new-product recommendation path verified where such products exist.


In [22]:
# Final validation

assert product_matrix.shape[0] == len(product_text)
assert len(item_to_index) == len(index_to_item)
assert content_result["HitRate@10"].between(0, 1).all()

print("Phase 3.2 validation: PASS")
print("Product representations:", f"{len(product_text):,}")
print("Evaluation users:", f"{evaluated:,}")
print("HitRate@10:", f"{content_hit_rate:.4f}")


Phase 3.2 validation: PASS
Product representations: 417,053
Evaluation users: 197
HitRate@10: 0.0355
